# Phase 6 — Decision Tree and Random Forest
Goal: test whether tree models beat the linear SVM (honest cost 45,100).
Preprocessing: no log, no scaling (trees don't need them).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Imports from src
from src.data_loader import load_aps_data
from src.preprocessing import (
    build_linear_preprocessor,
    build_tree_preprocessor,
    make_cv_splitter,
)
from src.evaluation import (
    calculate_aps_cost,
    calculate_classification_metrics,
    generate_oof_predictions,
    best_cost_threshold,
    oof_scores,
    honest_cost,
    robust_cv_predictions,
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


In [2]:
# Define paths and load the dataset
RES = PROJECT_ROOT / "reports" / "results"
RES.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "dataset"
X, y = load_aps_data(DATA_DIR / "aps_failure_training_set.csv")
cv = make_cv_splitter()

print(f"Loaded dataset: X shape = {X.shape}, y distribution = {dict(y.value_counts())}")


Loaded dataset: X shape = (60000, 170), y distribution = {0: np.int64(59000), 1: np.int64(1000)}


### Hypotheses & Predictions (before running):
1. The best single decision tree will cost more than the linear SVM baseline (45,100) due to high variance and axis-aligned step boundaries.
2. Random Forest ensembles will significantly reduce variance, improve probability calibration, and lower the honest misclassification cost.

In [3]:
import numpy as np
import pandas as pd

def make_tree(max_depth, min_leaf, class_weight=None):
    return Pipeline([
        ("prep", build_tree_preprocessor()),
        ("tree", DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_leaf=min_leaf,
            class_weight=class_weight,
            random_state=42
        )),
    ])

tree_rows, tree_scores = [], {}
for depth in [3, 5, 8, 12, None]:
    for leaf in [5, 50]:
        s, _, secs = oof_scores(make_tree(depth, leaf), X, y, cv)
        tree_scores[(depth, leaf)] = s
        tuned = best_cost_threshold(y, s)
        h_cost, h_fp, h_fn = honest_cost(y, s, cv)
        tree_rows.append({
            "model": "DecisionTree",
            "max_depth": depth,
            "min_samples_leaf": leaf,
            "seconds": round(secs, 1),
            "distinct_scores": len(np.unique(s)),
            "roc_auc": roc_auc_score(y, s),
            "pr_auc": average_precision_score(y, s),
            "tuned_cost": tuned["aps_cost"],
            "honest_cost": h_cost,
            "honest_fp": h_fp,
            "honest_fn": h_fn,
        })
        print(f"depth={depth} leaf={leaf} -> honest {h_cost} ({secs:.1f}s)")

tree_results = pd.DataFrame(tree_rows).sort_values("honest_cost").reset_index(drop=True)
tree_results.to_csv(RES / "phase6_tree_results.csv", index=False)
tree_results.round(4)


depth=3 leaf=5 -> honest 91540 (93.9s)
depth=3 leaf=50 -> honest 91540 (180.0s)
depth=5 leaf=5 -> honest 90280 (77.5s)
depth=5 leaf=50 -> honest 77630 (79.4s)
depth=8 leaf=5 -> honest 86800 (91.1s)
depth=8 leaf=50 -> honest 59520 (484.3s)
depth=12 leaf=5 -> honest 97120 (2366.8s)
depth=12 leaf=50 -> honest 64870 (31276.8s)
depth=None leaf=5 -> honest 115480 (97.2s)
depth=None leaf=50 -> honest 65630 (82.6s)


,model,max_depth,min_samples_leaf,seconds,distinct_scores,roc_auc,pr_auc,tuned_cost,honest_cost,honest_fp,honest_fn
0,DecisionTree,8.0,50,484.3,134,0.9627,0.7177,58920,59520,2652,66
1,DecisionTree,12.0,50,31276.8,154,0.9463,0.7182,63550,64870,2437,81
2,DecisionTree,NaN,50,82.6,161,0.9587,0.7209,63540,65630,2763,76
3,DecisionTree,5.0,50,79.4,81,0.9396,0.6816,77090,77630,1763,120
4,DecisionTree,8.0,5,91.1,79,0.9053,0.6952,86710,86800,1630,141
5,DecisionTree,5.0,5,77.5,68,0.9188,0.6772,85000,90280,1228,156
6,DecisionTree,3.0,5,93.9,38,0.9207,0.6259,91120,91540,1254,158
7,DecisionTree,3.0,50,180.0,40,0.9206,0.6130,91120,91540,1254,158
8,DecisionTree,12.0,5,2366.8,61,0.8808,0.6630,96200,97120,1962,155
9,DecisionTree,NaN,5,97.2,28,0.8892,0.6385,115030,115480,698,217


## Random Forest Benchmark
Ensemble trees with feature subsampling and bootstrap aggregating.

In [4]:
def make_rf(n_estimators=100, max_depth=None, min_leaf=5, class_weight="balanced"):
    return Pipeline([
        ("prep", build_tree_preprocessor()),
        ("rf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_leaf,
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1
        )),
    ])

# Random Forest Benchmark
rf_scores, _, rf_secs = oof_scores(make_rf(n_estimators=100, max_depth=12, min_leaf=5), X, y, cv)
rf_tuned = best_cost_threshold(y, rf_scores)
rf_cost, rf_fp, rf_fn = honest_cost(y, rf_scores, cv)

print(f"Random Forest (100 trees, depth 12): honest cost = {rf_cost} (FP={rf_fp}, FN={rf_fn}, {rf_secs:.1f}s)")


Random Forest (100 trees, depth 12): honest cost = 43230 (FP=2273, FN=41, 49.5s)


Result 1: best tree 59,520, higher than 45,100. Hypothesis held.
Observation: min_samples_leaf mattered more than depth; leaf 50 beat leaf 5 at every depth of 5 or more. The leaf-50 setting is at the grid's edge.

In [5]:
more_rows = []
for depth in [8, 12]:
    for leaf in [100, 200, 500]:
        s, _, secs = oof_scores(make_tree(depth, leaf), X, y, cv)
        tree_scores[(depth, leaf)] = s
        tuned = best_cost_threshold(y, s)
        h_cost, h_fp, h_fn = honest_cost(y, s, cv)
        more_rows.append({"model": "DecisionTree", "max_depth": depth,
                          "min_samples_leaf": leaf, "seconds": round(secs, 1),
                          "distinct_scores": len(np.unique(s)),
                          "roc_auc": roc_auc_score(y, s),
                          "pr_auc": average_precision_score(y, s),
                          "tuned_cost": tuned["aps_cost"],
                          "honest_cost": h_cost, "honest_fp": h_fp, "honest_fn": h_fn})
        print(f"depth={depth} leaf={leaf} -> honest {h_cost} ({secs:.0f}s)")

tree_results = (pd.concat([tree_results, pd.DataFrame(more_rows)], ignore_index=True)
                  .sort_values("honest_cost").reset_index(drop=True))
tree_results.to_csv(RES / "phase6_tree_results.csv", index=False)
tree_results.round(4).head(8)

depth=8 leaf=100 -> honest 66300 (51s)
depth=8 leaf=200 -> honest 60080 (122s)
depth=8 leaf=500 -> honest 61410 (127s)
depth=12 leaf=100 -> honest 64450 (60s)
depth=12 leaf=200 -> honest 58580 (59s)
depth=12 leaf=500 -> honest 62460 (57s)


,model,max_depth,min_samples_leaf,seconds,distinct_scores,roc_auc,pr_auc,tuned_cost,honest_cost,honest_fp,honest_fn
0,DecisionTree,12.0,200,59.4,91,0.9644,0.6356,57380,58580,3158,54
1,DecisionTree,8.0,50,484.3,134,0.9627,0.7177,58920,59520,2652,66
2,DecisionTree,8.0,200,122.5,79,0.9599,0.6341,58880,60080,3108,58
3,DecisionTree,8.0,500,127.1,54,0.9709,0.4557,57010,61410,3341,56
4,DecisionTree,12.0,500,56.7,60,0.9716,0.4554,58060,62460,3346,58
5,DecisionTree,12.0,100,60.0,114,0.9478,0.6876,62890,64450,3045,68
6,DecisionTree,12.0,50,31276.8,154,0.9463,0.7182,63550,64870,2437,81
7,DecisionTree,NaN,50,82.6,161,0.9587,0.7209,63540,65630,2763,76


In [6]:
import time
tr, va = next(iter(cv.split(X, y)))
t0 = time.time()
make_rf(n_estimators=100, max_depth=None, min_leaf=20, class_weight=None).fit(X.iloc[tr], y.iloc[tr])
print("one fold, 100 trees:", round(time.time() - t0, 1), "seconds")

one fold, 100 trees: 11.6 seconds


In [7]:
rf_rows, rf_scores = [], {}
for leaf in [5, 20, 50]:
    s, _, secs = oof_scores(make_rf(n_estimators=100, max_depth=None, min_leaf=leaf,
                                    class_weight=None), X, y, cv)
    rf_scores[leaf] = s
    tuned = best_cost_threshold(y, s)
    h_cost, h_fp, h_fn = honest_cost(y, s, cv)
    rf_rows.append({"model": "RandomForest", "n_estimators": 100, "min_samples_leaf": leaf,
                    "class_weight": None, "seconds": round(secs, 1),
                    "distinct_scores": len(np.unique(s)),
                    "roc_auc": roc_auc_score(y, s),
                    "pr_auc": average_precision_score(y, s),
                    "tuned_cost": tuned["aps_cost"],
                    "honest_cost": h_cost, "honest_fp": h_fp, "honest_fn": h_fn})
    print(f"leaf={leaf} -> honest {h_cost} ({secs:.0f}s)")

rf_results = pd.DataFrame(rf_rows).sort_values("honest_cost").reset_index(drop=True)
rf_results.to_csv(RES / "phase6_rf_results.csv", index=False)
rf_results.round(4)

leaf=5 -> honest 44220 (71s)
leaf=20 -> honest 47520 (70s)
leaf=50 -> honest 52260 (67s)


,model,n_estimators,min_samples_leaf,class_weight,seconds,distinct_scores,roc_auc,pr_auc,tuned_cost,honest_cost,honest_fp,honest_fn
0,RandomForest,100,5,None,71.2,6030,0.9853,0.8563,42290,44220,2222,44
1,RandomForest,100,20,None,69.6,12966,0.9844,0.8119,46030,47520,2902,37
2,RandomForest,100,50,None,66.5,21267,0.9849,0.7617,50070,52260,3476,35


In [8]:
test_a_configs = [
    {"n_estimators": 100, "min_samples_leaf": 1},
    {"n_estimators": 100, "min_samples_leaf": 2},
    {"n_estimators": 300, "min_samples_leaf": 5},
]

In [9]:
for config in test_a_configs:
    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        min_samples_leaf=config["min_samples_leaf"],
        class_weight=None,
        random_state=42,
        n_jobs=-1,
    )

    # Use the SAME tree preprocessing pipeline,
    # OOF generation, threshold search, and honest-cost
    # functions already used for your existing:
    # leaf=5, leaf=20, leaf=50 experiments.

In [10]:
scores_dir = PROJECT_ROOT / "models" / "oof_scores"
scores_dir.mkdir(parents=True, exist_ok=True)
for leaf, s in rf_scores.items():
    np.save(scores_dir / f"rf_leaf{leaf}.npy", s)
np.save(scores_dir / "tree_d12_leaf200.npy", tree_scores[(12, 200)])